# 1. What exactly is a pretrained model?

## A pretrained language model is a neural network whose parameters (weights and biases) have already been learned through training on a large dataset.

For example, a pretrained language model contains:
- An embedding layer.
- Transformer layers.
- Attention mechanisms.
- Feed-forward networks.
- Normalization layers.
- An output projection or language-model head, depending on the architecture.
The model's learned parameters are stored in weight files, commonly using formats such as:
- .safetensors
- .bin (PyTorch serialization)
The model repository also typically contains a configuration file describing the architecture.

A useful distinction:
```
Component                                         	Responsibility
Tokenizer	                                     Converts text into token IDs
Model configuration	                           Describes the model architecture
Model weights	                                 Store learned parameters
Model class	                              Defines how the architecture operates
Generation configuration	          Stores default text-generation settings, when provided
```

Hugging Face provides from_pretrained() to load models from the Hub or a local directory.

Different Auto classes support different tasks:
```
Class	                                              Purpose
AutoModel	                          Base model without a task-specific head
AutoModelForCausalLM	                       Text generation
AutoModelForSeq2SeqLM	                     Encoder-decoder generation
AutoModelForSequenceClassification	       Text classification
AutoModelForMaskedLM	                     Masked language modeling
````
For our LLM fine-tuning journey, we'll primarily use AutoModelForCausalLM.

In [3]:
import torch

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM
)

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

In [5]:
config = AutoConfig.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("model type :", config.model_type)
print("hidden size :", config.hidden_size)
print("layers :", config.num_hidden_layers)
print("Attention head :", config.num_attention_heads)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model type : qwen2
hidden size : 896
layers : 24
Attention head : 14


Explanation
- AutoConfig loads architecture information without downloading the full model weights.
- AutoTokenizer loads the tokenizer associated with the model.

In [6]:
if torch.cuda.is_available():
    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
    device_map = "auto"
else:
    dtype = torch.float32
    device_map = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map=device_map,
    low_cpu_mem_usage=True
)

model.eval()

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


Explanation
- torch_dtype specifies the precision used for model weights.
- device_map controls where model components are placed.
- low_cpu_mem_usage=True helps reduce peak CPU memory consumption.
- model.eval() puts the model into evaluation mode.
For reproducibility, we can later add a pinned repository revision and other loading options.

3. Precision and hardware
Notes

Precision determines how many bits are used to represent each model parameter.
Format	Bits	Memory per parameter
FP32	32	4 bytes
FP16	16	2 bytes
BF16	16	2 bytes
INT8	8	1 byte
INT4	4	0.5 byte

- FP32: Higher precision, more memory.
- FP16: Lower memory, widely used for GPU inference.
- BF16: Similar memory to FP16, with a wider exponent range.
- INT8/INT4: Quantized representations that reduce weight memory.
A rough estimate:
\[
\text{Weight memory} =
\text{Parameters} \times \text{Bytes per parameter}
\]

For a 500-million-parameter model, FP16 weights require approximately 1 GB of memory.

Actual memory usage is higher due to activations, KV cache and runtime overhead.

Important hardware concepts
- CPU: General-purpose processor.
- GPU: Accelerates parallel neural-network computations.
- VRAM: Memory available on the GPU.
- device_map="auto": Automatically distributes model components according to available devices and memory.

In [7]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB




---

---





4. Model inspection and basic inference
Notes

After loading a model, inspect its parameters and architecture to understand what you are working with.

In [9]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model class:", model.__class__.__name__)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("Data type:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)

Model class: Qwen2ForCausalLM
Total parameters: 494,032,768
Trainable parameters: 494,032,768
Data type: torch.bfloat16
Device: cuda:0


In [12]:
messages = [
    {
        "role": "user",
        "content": "Explain machine learning in simple terms."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

input_device = model.get_input_embeddings().weight.device
inputs = {k: v.to(input_device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Machine learning is a subset of artificial intelligence that involves the development and application of algorithms to enable computers to learn from data without being explicitly programmed. In simpler terms, it's about training machines to make decisions or predictions based on patterns and rules learned from historical data.

Here’s how it works:

1. **Data Collection**: The first step is collecting a large dataset that includes various features (inputs) and corresponding outputs (labels).

2. **Feature Engineering**: This involves transforming raw data into useful inputs for


Explanation
- apply_chat_template() formats the conversation for the model.
- generate() generates new tokens.
- max_new_tokens limits response length.
- temperature controls sampling randomness.
- top_p controls nucleus sampling.
- inference_mode() disables gradient tracking.
We'll study generation strategies in detail in Notebook 05.



---



---



5. Saving and reusable model loading
Notes

Hugging Face provides save_pretrained() to save model weights and configuration, and the tokenizer can be saved separately.

This allows us to reload the model without downloading it again.

In [13]:
save_path = "/content/drive/MyDrive/qwen_model"

model.save_pretrained(
    save_path,
    safe_serialization=True
)

tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/qwen_model/tokenizer_config.json',
 '/content/qwen_model/chat_template.jinja',
 '/content/qwen_model/tokenizer.json')

In [15]:
save_path = "/content/drive/MyDrive/qwen_model"
tokenizer = AutoTokenizer.from_pretrained(save_path)

model = AutoModelForCausalLM.from_pretrained(
    save_path,
    torch_dtype=dtype,
    device_map="auto"
)

print("Reloaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Reloaded successfully!
